# Défi 2 NLP (medRxiv 2026) — Projet 1 : Discipline Prediction
**Approche :** Deep Learning (Fine-tuning de `PubMedBERT` avec perte pondérée)  
**Auteur :** Redouane Kenniche

---

### Objectif du notebook
Ce notebook met en œuvre un modèle neuronal spécialisé dans la littérature biomédicale (`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract`) pour classer les résumés (*abstracts*) de preprints medRxiv parmi les **16 disciplines médicales** (`category_grp`), en optimisant la métrique officielle **F1 macro**.

## 1. Importation des bibliothèques et configuration matérielle
Chargement de PyTorch, Hugging Face Transformers et Scikit-learn, puis détection automatique du GPU (`cuda`) pour accélérer l'entraînement car temps de l'entraînement assez conséquent selon les machines.

In [1]:
import html
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
print(f"Périphérique utilisé : {DEVICE}")

Périphérique utilisé : cuda


## 2. Fonctions de chargement et de nettoyage des textes
Utilitaires pour localiser les fichiers CSV (compressés ou non) avec le bon séparateur (`;` ou `,`) et nettoyer les balises HTML résiduelles tout en conservant les symboles mathématiques et statistiques.

In [2]:
def trouver_fichier(noms_possibles):
    for nom in noms_possibles:
        if os.path.exists(nom):
            return nom
    return None


def lire_csv_robuste(filepath):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        for sep in [";", ","]:
            try:
                df = pd.read_csv(filepath, encoding=enc, sep=sep)
                df.columns = df.columns.str.strip().str.strip('"')
                if "abstract" in {c.lower() for c in df.columns}:
                    return df
            except Exception:
                continue
    raise ValueError(f"Impossible de lire : {filepath}")


def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte)
    # Suppression ciblée des balises HTML uniquement
    texte = re.sub(
        r"</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>",
        " ",
        texte,
    )
    return re.sub(r"\s+", " ", texte).strip()

## 3. Préparation, dédoublonnage et découpage stratifié
Chargement des données d'entraînement, suppression des versions dupliquées d'un même preprint pour éviter toute fuite entre l'entraînement et la validation, encodage des 16 disciplines (`category_grp`) et séparation stratifiée (80 % entraînement / 20 % validation).

In [3]:
TRAIN_FILE = trouver_fichier([
    "ClassificationsMedicales_AbstractsTrainingSet.csv",
    "ClassificationsMedicales_AbstractsTrainingSet.csv.gz",
])
TEST_FILE = trouver_fichier([
    "ClassificationsMedicales_AbstractsTestSet.csv",
    "ClassificationsMedicales_AbstractsTestSet.csv.gz",
])

df_train = lire_csv_robuste(TRAIN_FILE)
col_map = {c.lower(): c for c in df_train.columns}
col_abstract = col_map["abstract"]
col_target = col_map.get("category_grp", col_map.get("category_group"))

df_train = df_train.dropna(subset=[col_abstract, col_target]).reset_index(
    drop=True
)
df_train["clean_abstract"] = df_train[col_abstract].apply(nettoyer_texte)
df_train[col_target] = df_train[col_target].astype(str).str.strip()

# Suppression des versions successives d'un même abstract
df_train = df_train.drop_duplicates(subset=["clean_abstract"]).reset_index(
    drop=True
)

labels_list = sorted(df_train[col_target].unique())
label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for i, l in enumerate(labels_list)}
df_train["label"] = df_train[col_target].map(label2id)
print(f"Classes détectées ({len(labels_list)}) | Lignes uniques : {len(df_train)}")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train["clean_abstract"].tolist(),
    df_train["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_train["label"].tolist(),
)

Classes détectées (16) | Lignes uniques : 9324


## 4. Tokenisation et création des Datasets PyTorch
Encodage des résumés avec le tokenizer de `PubMedBERT` (troncature à 384 tokens) et calcul des poids inverses des fréquences de classes pour compenser le déséquilibre entre disciplines.

In [4]:
# Pondération des classes pour optimiser le F1 macro
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(labels_list)),
    y=train_labels,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class AbstractDataset(Dataset):

    def __init__(self, texts, labels=None, max_len=384):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True, max_length=max_len
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])


train_ds = AbstractDataset(train_texts, train_labels)
val_ds = AbstractDataset(val_texts, val_labels)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

## 5. Architecture du Trainer pondéré et entraînement du modèle
Personnalisation de la fonction de perte (`CrossEntropyLoss` pondérée), définition des métriques d'évaluation (`f1_macro` et `accuracy`), puis fine-tuning de `PubMedBERT` sur 4 époques avec sauvegarde automatique du meilleur modèle.

In [5]:
class WeightedTrainer(Trainer):

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels_list),
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="./results_projet1",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
eval_res = trainer.evaluate()
print(
    f"Validation PubMedBERT | F1 macro : {eval_res['eval_f1_macro']:.4f} |"
    f" Accuracy : {eval_res['eval_accuracy']:.4f}"
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.420753,1.160832,4,0.614497,0.626810


Validation PubMedBERT | F1 macro : 0.6145 | Accuracy : 0.6268


## 6. Inférence sur le jeu de test et export des prédictions
Application du meilleur modèle entraîné sur les abstracts du jeu de test et génération du fichier final au format attendu `(ID, pred)`.

In [6]:
if TEST_FILE:
    df_test = lire_csv_robuste(TEST_FILE)
    test_col_map = {c.lower(): c for c in df_test.columns}
    col_id_test = test_col_map.get("id", df_test.columns[0])
    col_abs_test = test_col_map["abstract"]

    test_ds = AbstractDataset(
        df_test[col_abs_test].fillna("").apply(nettoyer_texte).tolist()
    )
    preds_logits = trainer.predict(test_ds).predictions
    preds_ids = np.argmax(preds_logits, axis=-1)

    df_sub = pd.DataFrame({
        "ID": df_test[col_id_test],
        "pred": [id2label[i] for i in preds_ids],
    })
    df_sub.to_csv("Projet1_Script2_DL.csv", index=False)
    print(f"Fichier généré : Projet1_Script2_DL.csv ({df_sub.shape})")

Fichier généré : Projet1_Script2_DL.csv ((1940, 2))
